# Solution - Exercise 1 - TravelMind Triage Pipeline

Worked solutions, built to be read by anyone. Each part follows the same shape:

- the idea in plain words
- a picture of what is actually moving
- the working code
- why it works
- the wrong turn people take

Part A runs with no AWS at all, it is pure Python plumbing. Parts C to E call Bedrock.

Run this once:

```python
```

In [1]:
from langchain_aws import ChatBedrockConverse
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.output_parsers import StrOutputParser
from langchain_core.runnables import RunnablePassthrough, RunnableParallel, RunnableLambda
from pydantic import BaseModel, Field

MODEL_ID = "us.anthropic.claude-haiku-4-5-20251001-v1:0"
llm = ChatBedrockConverse(model_id=MODEL_ID, region_name="us-east-1", temperature=0.2)

## Part A - Trace it

**The idea in plain words.** A LangChain chain is a conveyor belt. A dict rides the belt. Each station either transforms the dict or bolts a new key onto it. Nothing is thrown away unless you throw it away.

**The picture.**

```mermaid
graph LR
    I0["input: complaint"] --> S1["dict map: add key issue"]
    S1 --> S2["assign: add key words"]
    S2 --> OUT["issue and words"]
```

Station 1 is `{"issue": ...}`. Station 2 is `.assign(words=...)`. The belt never loses `issue`, so by the end you hold both keys.

In [2]:
step = (
    {"issue": lambda x: x["complaint"].strip().lower()}
    | RunnablePassthrough.assign(words=lambda x: len(x["issue"].split()))
)

print(step.invoke({"complaint": "  BAG Lost at DEL  "}))

{'issue': 'bag lost at del', 'words': 4}


**The answer.**

| Stage | Keys present | `issue` value |
|---|---|---|
| after the dict map | `issue` | `bag lost at del` |
| after `.assign(words=...)` | `issue`, `words` | `bag lost at del`, `words=4` |

Printed dict: `{'issue': 'bag lost at del', 'words': 4}`

`strip().lower()` cleans the text, then `split()` on `bag lost at del` gives four words.

**Why `x` is the whole dict.** A plain dict written inside a chain quietly becomes a `RunnableParallel`. Every value in that dict receives the same input, which is the full dict on the belt. So the lambda reads `x["complaint"]`, not a lone string.

**The wrong turn.** Rename the input key from `complaint` to `text` but leave the lambda reading `x["complaint"]`, and station 1 raises `KeyError: 'complaint'`. The station never got the key it asked for.

## Part B - Debug and fix

Three snippets, one bug each. Spot the line, fix in one move.

### B1

```python
chain = ChatPromptTemplate.from_messages([("human", "{q}")]) | llm | StrOutputParser
```

- **Broken line.** The last one. `StrOutputParser` is the class itself, not an object.
- **Plain why.** The pipe hands data to each stage and calls it. A class is a blueprint, not a worker. You gave it the blueprint.
- **Fix.** Add the parentheses so you build one: `StrOutputParser()`.

### B2

```python
fare=ChatPromptTemplate.from_messages([("human", "Fare handling for: {text}")]) | llm | StrOutputParser()
gather.invoke({"brief": "JX48Q2 cancelled, Gold tier"})
```

- **Broken line.** The `fare` branch asks for `{text}`. The input only carries `brief`.
- **Plain why.** A template variable is a promise to supply that key. No `text` key arrives, so it raises `KeyError: 'text'`.
- **Fix.** Match the key: `("human", "Fare handling for: {brief}")`.

### B3

```python
routes = {"rebooking": rebooking_chain, "refund": refund_chain}

def route(inp):
    category = classifier.invoke({"request": inp["request"]}).category
    return routes[category]
```

- **Two defects.** `routes[category]` explodes with `KeyError` for any category outside the two keys. And it returns a chain object without ever running it.
- **Plain why.** `dict[key]` has no mercy for a missing key, and returning a runnable is not the same as invoking it.
- **Fix, one line.**

```python
    return routes.get(category, quick).invoke(inp)
```

`get` supplies a safe default, and `.invoke(inp)` actually runs the chosen chain.

## Part C - Diagram to code

**The idea in plain words.** Three questions have nothing to do with each other, so ask them at the same time, then hand all three answers to a writer that stitches them into one reply.

**The picture.**

```mermaid
graph TD
    IN["brief"] --> P["fan out"]
    P --> A["policy in 2 lines"]
    P --> B["fare handling in 2 lines"]
    P --> C["two rebooking options"]
    A --> S["synthesize one reply"]
    B --> S
    C --> S
    S --> OUT["reply"]
```

In [3]:
policy = ChatPromptTemplate.from_messages([
    ("system", "State the Gold-tier cancellation policy in 2 lines."),
    ("human", "{brief}"),
]) | llm | StrOutputParser()

fare = ChatPromptTemplate.from_messages([
    ("system", "Explain fare-difference handling for an involuntary change in 2 lines."),
    ("human", "{brief}"),
]) | llm | StrOutputParser()

options = ChatPromptTemplate.from_messages([
    ("system", "List two concrete rebooking options."),
    ("human", "{brief}"),
]) | llm | StrOutputParser()

gather = RunnableParallel(policy=policy, fare=fare, options=options)

synth = ChatPromptTemplate.from_messages([
    ("system", "Merge the notes into one calm passenger reply."),
    ("human", "policy: {policy}\nfare: {fare}\noptions: {options}"),
]) | llm | StrOutputParser()

pipeline = gather | synth

print(pipeline.invoke({"brief": "PNR JX48Q2, BLR to DEL cancelled, Gold tier."}))

# Your Rebooking Options – Gold Tier Benefits Applied

Thank you for your patience. As a Gold-tier passenger, we're pleased to confirm that your cancelled flight **JX48Q2 (BLR-DEL)** qualifies for our premium rebooking benefits.

## Your Options

**Option 1: Direct Flight (Recommended)**
Rebook on the next available BLR-DEL flight at no additional cost. Your seat selection and meal preferences will be retained, and you'll receive priority placement on your preferred time slot.

**Option 2: Alternative Route**
If direct flights are full, we can rebook you via Mumbai (BOM) with a same-day connection. You'll enjoy complimentary lounge access during your layover, with a total journey time of 5-6 hours.

## What's Included
- **No fare difference charges** – Gold tier covers any price variance
- **No additional fees** – Both options are fully covered
- **Full refund available** – If neither option works for you

## Next Steps
Please contact our Gold Tier support line or visit the airport cou

**Why `gather | synth` needs no glue.**

```mermaid
graph LR
    G["gather output"] --> K1["policy"]
    G --> K2["fare"]
    G --> K3["options"]
    K1 --> SY["synth inputs"]
    K2 --> SY
    K3 --> SY
```

`gather` returns a dict with keys `policy`, `fare`, `options`. `synth` asks for exactly those three. The output shape of one is the input shape of the next, so the pipe clicks with no adapter in between.

**The wrong turn.** Rename a branch to `pol` while `synth` still reads `{policy}`, and the join breaks. The pipe clicks only when both sides agree on the key names.

## Part D - Convert plain Python to LCEL

**The idea in plain words.** The boto3 version is three separate phone calls with sticky notes passed by hand between them. LCEL folds the same three calls into one machine you press once.

**The picture.**

```mermaid
graph TD
    subgraph Before["boto3 hand rolled"]
        b1["call extract"] --> b2["call classify"]
        b2 --> b3["call draft"]
    end
    subgraph After["LCEL one chain"]
        a1["extract"] --> a2["assign urgency"]
        a2 --> a3["draft"]
    end
```

In [4]:
extract = ChatPromptTemplate.from_messages([
    ("system", "Extract the core issue in one sentence."),
    ("human", "{complaint}"),
]) | llm | StrOutputParser()

classify = ChatPromptTemplate.from_messages([
    ("system", "Urgency as low, medium, or high. Reply with only the label."),
    ("human", "Issue: {issue}"),
]) | llm | StrOutputParser()

draft = ChatPromptTemplate.from_messages([
    ("system", "Draft a TravelMind reply matched to the urgency."),
    ("human", "Issue: {issue}\nUrgency: {urgency}"),
]) | llm | StrOutputParser()

triage = (
    {"issue": extract}
    | RunnablePassthrough.assign(urgency=classify)
    | draft
)

print(triage.invoke({
    "complaint": "My BLR to DEL flight JX48Q2 was cancelled and I have a wedding tonight."
}))

# TravelMind Response: Flight Cancellation - Wedding Emergency

I understand this is extremely stressful. Let me help you navigate this systematically.

## IMMEDIATE ACTIONS (Next 15-20 minutes)

**1. Call Your Airline NOW**
- Have your booking reference ready
- Explain: "My flight is cancelled and I have a wedding tonight in Delhi"
- Request: Emergency rebooking on ANY flight to DEL today
- Ask: "What's the earliest available option you can confirm?"
- Get: Confirmation number and flight details in writing (email/SMS)

**2. Parallel Search - Don't Wait for Airline**
Open these simultaneously:
- **MakeMyTrip/Skyscanner** - filter BLR→DEL, today, sort by departure time
- **Airline websites directly** - IndiGo, Air India, SpiceJet, Vistara
- Look at flights departing in next 2-6 hours

**3. Expand Your Options**
- BLR→Mumbai→DEL (if direct flights full)
- BLR→Hyderabad→DEL (check timing)
- Check if any evening/night flights still available

---

## CRITICAL INFO NEEDED

To give you more 

**How the belt fills up.**

| Stage | Belt before | Belt after |
|---|---|---|
| `{"issue": extract}` | `{complaint}` | `{issue}` |
| `.assign(urgency=classify)` | `{issue}` | `{issue, urgency}` |
| `draft` | `{issue, urgency}` | the reply |

`assign` is the move worth keeping. It runs `classify`, then bolts `urgency` next to `issue` instead of replacing it. That is how a later step sees everything the earlier steps produced.

**What you gained for free.** The same three model calls, but now `triage.batch([...])` runs a hundred complaints concurrently and `triage.stream(...)` gives token-by-token output. Zero extra lines for either. The boto3 version needs threads and manual streaming to match that.

## Part E - Build the front door

**The idea in plain words.** Do not run an expensive three-question gather for someone who just asked when boarding starts. Check what they want first, then send only the heavy cases down the heavy path.

**The picture.**

```mermaid
graph TD
    IN["brief"] --> C["classify intent"]
    C --> Q{"rebooking or refund?"}
    Q -->|"yes"| P["full pipeline: gather then synth"]
    Q -->|"no"| K["one-line acknowledgement"]
```

In [5]:
class Intent(BaseModel):
    category: str = Field(description="one of: rebooking, refund, other")

classifier = ChatPromptTemplate.from_messages([
    ("system", "Classify the request into exactly one of: rebooking, refund, other."),
    ("human", "{brief}"),
]) | llm.with_structured_output(Intent)

quick = ChatPromptTemplate.from_messages([
    ("system", "Give a one-line acknowledgement to the passenger."),
    ("human", "{brief}"),
]) | llm | StrOutputParser()

def front_door(inp):
    category = classifier.invoke(inp).category
    if category in ("rebooking", "refund"):
        return pipeline.invoke(inp)
    return quick.invoke(inp)

router = RunnableLambda(front_door)

for brief in [
    "Move JX48Q2 to the morning flight.",
    "What time is boarding, roughly?",
]:
    print(router.invoke({"brief": brief}))
    print("---")

# Help with Your Flight Change Request

I appreciate you wanting to move reservation JX48Q2 to a morning flight. Unfortunately, I don't have access to booking systems, so I can't make this change directly for you.

Here's what you can do:

**Self-Service Options:**
- Log into the airline's website or mobile app with your booking reference (JX48Q2) and use the "Manage My Booking" or "Change Flight" feature to view available morning departures
- Contact the airline's customer service team directly by phone or through their website

**Good News About Costs:**
If this is an involuntary change on the airline's part, you typically won't pay any additional fare difference if the morning flight is in the same cabin class. If the morning flight is cheaper, you may receive a refund or credit for the difference.

If you booked through a travel agent, they can also assist with the rebooking.

Is there anything else I can help clarify about airline policies or procedures?
---
I don't have informati

**Why the branches swap cleanly.** `pipeline`, `quick`, and `classifier` all read the same `{"brief": ...}` input. Same input shape means you can point the router at any of them without rewiring.

**Cost intuition.** Count the model calls.

| Path | Calls |
|---|---|
| classify then one-line | 2 |
| classify then full pipeline | 1 + 3 + 1 = 5 |

Routing first adds one classify call to every request. It earns that call back the moment a cheap request skips the four-call gather. On a real support queue where most messages are simple, the trade pays for itself fast.

**Skeptic's question.** The whole saving rests on the classifier being right. What happens to a rebooking wrongly tagged `other`? It gets a one-line brush-off instead of real help. Log the category, watch the misroute rate, and decide if the saving is worth the occasional bad call. A cheaper pipeline that quietly fails the important cases is not cheaper.

## Recap

```mermaid
graph LR
    A["trace: dict on a belt"] --> B["debug: read the error, one-line fix"]
    B --> C["diagram to code: keys must line up"]
    C --> D["boto3 to LCEL: assign carries state"]
    D --> E["route first: pay one call to skip four"]
```

One thread runs through all five. In a workflow, **you** hold the control flow, and every join works because the shapes on both sides agree.